# CMP01 anchored ordered residual core

factor = 0.70*CMP01 + 0.15*residual(triple) + 0.15*residual(F01_WT17)

The core contains no order-book flow leg.

All legs use T-day or earlier data. The factor is formed after the close for T+1 evaluation. Weights and axis order are fixed and are not fitted inside this notebook.

In [ ]:
"""
因子名：共振承接双机制等权（F01_WT17_EQ）
【市场机理】F01 衡量个股日内收益对市场共振的独立偏离，WT17 衡量大成交额分钟的
价格冲击能否被市场深度吸收。二者分别在同日截面转为中心秩后 50/50 等权，避免量纲
差异，也不利用回测结果拟合权重。
【冻结公式】factor = 0.5 * rank_center(F01_RESONANCE_V1)
                    + 0.5 * rank_center(WT17_ABSORPTION_V1)。
只用 T 日及以前数据，T+1 使用。
"""

import dai
import numpy as np
import pandas as pd


KEY = ["date", "instrument"]
F01_LOOKBACK_CALENDAR_DAYS = 90


def _day(values):
    out = pd.to_datetime(values, errors="coerce")
    if getattr(out.dt, "tz", None) is not None:
        out = out.dt.tz_localize(None)
    return out.dt.normalize()


def _bar1m(datasources):
    if not isinstance(datasources, dict) or not isinstance(datasources.get("bar1m"), str):
        raise TypeError("datasources must provide a string bar1m table name")
    return datasources["bar1m"]


def _pool(start, end):
    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")]},
        compression=True,
    ).df()
    if pool.empty:
        return pd.DataFrame(columns=KEY)
    pool["date"] = _day(pool["date"])
    pool["instrument"] = pool["instrument"].astype(str)
    return pool.dropna().drop_duplicates(KEY)


def _f01_pool(start, end):
    parts = []
    stop = end.normalize() + pd.Timedelta(days=1)
    cursor = start.normalize().replace(day=1)
    while cursor < stop:
        nxt = cursor + pd.DateOffset(months=1)
        lo, hi = max(cursor, start.normalize()), min(nxt, stop)
        pool = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={"date": [lo.strftime("%Y-%m-%d"), hi.strftime("%Y-%m-%d")]},
            compression=True,
        ).df()
        if not pool.empty:
            pool["date"] = _day(pool["date"])
            pool["instrument"] = pool["instrument"].astype(str)
            parts.append(pool.loc[(pool["date"] >= lo) & (pool["date"] < hi), KEY])
        cursor = nxt
    if not parts:
        return pd.DataFrame(columns=KEY)
    return pd.concat(parts, ignore_index=True).dropna().drop_duplicates(KEY)


def _weekly_chunks(start, end):
    cursor = pd.Timestamp(start).normalize()
    finish = pd.Timestamp(end).normalize() + pd.Timedelta(days=1)
    while cursor < finish:
        nxt = min(cursor + pd.Timedelta(days=7), finish)
        yield cursor, nxt
        cursor = nxt


def _f01_read_bars(datasources, pool, start, end):
    ids = pool["instrument"].dropna().astype(str).unique().tolist()
    parts = []
    for lo, hi in _weekly_chunks(start, end):
        raw = dai.query(
            "SELECT date, instrument, open, close FROM " + _bar1m(datasources) + " "
            "WHERE EXTRACT(HOUR FROM date) * 100 + EXTRACT(MINUTE FROM date) >= 932 "
            "AND EXTRACT(HOUR FROM date) * 100 + EXTRACT(MINUTE FROM date) <= 1456",
            filters={"date": [lo.strftime("%Y-%m-%d"), hi.strftime("%Y-%m-%d")], "instrument": ids},
            compression=True,
        ).df()
        if raw.empty:
            continue
        raw["minute"] = pd.to_datetime(raw["date"], errors="coerce")
        if getattr(raw["minute"].dt, "tz", None) is not None:
            raw["minute"] = raw["minute"].dt.tz_localize(None)
        raw = raw[(raw["minute"] >= lo) & (raw["minute"] < hi)].copy()
        raw["date"] = _day(raw["minute"])
        raw["instrument"] = raw["instrument"].astype(str)
        parts.append(raw)
    if not parts:
        return pd.DataFrame(columns=KEY + ["minute", "open", "close"])
    return pd.concat(parts, ignore_index=True).merge(pool, on=KEY, how="inner")


def _f01_daily_intraday_return(raw):
    if raw.empty:
        return pd.DataFrame(columns=KEY + ["intraday_return"])
    raw = raw.copy()
    raw["open"] = pd.to_numeric(raw["open"], errors="coerce")
    raw["close"] = pd.to_numeric(raw["close"], errors="coerce")
    raw = raw.loc[
        np.isfinite(raw["open"]) & (raw["open"] > 0)
        & np.isfinite(raw["close"]) & (raw["close"] > 0)
    ].sort_values(KEY + ["minute"])
    if raw.empty:
        return pd.DataFrame(columns=KEY + ["intraday_return"])
    daily = raw.groupby(KEY, sort=False, observed=True).agg(
        first_open=("open", "first"), last_close=("close", "last")
    ).reset_index()
    daily["intraday_return"] = daily["last_close"] / daily["first_open"] - 1.0
    return daily.loc[np.isfinite(daily["intraday_return"]), KEY + ["intraday_return"]]


def _f01_factor(pool, daily, start, end):
    work = pool.merge(daily, on=KEY, how="left").sort_values(KEY)
    market = work.groupby("date", sort=False, observed=True)["intraday_return"].transform("mean")
    work["raw"] = (work["intraday_return"] - market).abs()
    group = work.groupby("date", sort=False, observed=True)["raw"]
    lo = group.transform(lambda x: x.quantile(0.01))
    hi = group.transform(lambda x: x.quantile(0.99))
    work["_clip"] = work["raw"].clip(lo, hi)
    count = work.groupby("date", sort=False, observed=True)["_clip"].transform("count")
    work["_clean"] = work.groupby("date", sort=False, observed=True)["_clip"].rank(method="average") / count - 0.5
    work = work.sort_values(["instrument", "date"])
    work["factor"] = -work.groupby("instrument", sort=False, observed=True)["_clean"].transform(
        lambda x: x.rolling(20, min_periods=10).mean()
    )
    return work.loc[
        (work["date"] >= start) & (work["date"] <= end) & np.isfinite(work["factor"]),
        KEY + ["factor"],
    ]


def _compute_f01(datasources, start, end):
    warm = start - pd.Timedelta(days=F01_LOOKBACK_CALENDAR_DAYS)
    pool = _f01_pool(warm, end)
    raw = _f01_read_bars(datasources, pool, warm, end + pd.Timedelta(days=1))
    return _f01_factor(pool, _f01_daily_intraday_return(raw), start, end)


def _months(start, end):
    stop = end + pd.Timedelta(days=1)
    cursor = start.replace(day=1)
    while cursor < stop:
        nxt = cursor + pd.DateOffset(months=1)
        yield max(cursor, start), min(nxt, stop)
        cursor = nxt


def _wt17_read_month(datasources, pool, start, end):
    month_pool = pool[(pool["date"] >= start) & (pool["date"] < end)]
    ids = month_pool["instrument"].dropna().astype(str).unique().tolist()
    if not ids:
        return pd.DataFrame(columns=KEY + ["minute", "close", "amount"])
    raw = dai.query(
        "SELECT date, instrument, close, amount FROM " + _bar1m(datasources) + " "
        "WHERE EXTRACT(HOUR FROM date) * 100 + EXTRACT(MINUTE FROM date) >= 932 "
        "AND EXTRACT(HOUR FROM date) * 100 + EXTRACT(MINUTE FROM date) <= 1456",
        filters={"date": [start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")], "instrument": ids},
        compression=True,
    ).df()
    if raw.empty:
        return pd.DataFrame(columns=KEY + ["minute", "close", "amount"])
    raw["minute"] = pd.to_datetime(raw["date"], errors="coerce")
    if getattr(raw["minute"].dt, "tz", None) is not None:
        raw["minute"] = raw["minute"].dt.tz_localize(None)
    raw = raw[(raw["minute"] >= start) & (raw["minute"] < end)].copy()
    raw["date"] = _day(raw["minute"])
    raw["instrument"] = raw["instrument"].astype(str)
    raw["close"] = pd.to_numeric(raw["close"], errors="coerce")
    raw["amount"] = pd.to_numeric(raw["amount"], errors="coerce")
    return raw.merge(month_pool, on=KEY, how="inner").sort_values(KEY + ["minute"])


def _wt17_one_day(frame):
    close = frame["close"].to_numpy(dtype=float)
    amount = frame["amount"].to_numpy(dtype=float)
    if close.size < 61 or not np.isfinite(close).all() or not np.isfinite(amount).all() or np.min(close) <= 0:
        return np.nan
    ret = np.abs(np.diff(np.log(close)))
    traded = amount[1:]
    valid = np.isfinite(ret) & np.isfinite(traded) & (traded > 0)
    ret, traded = ret[valid], traded[valid]
    if ret.size < 60:
        return np.nan
    impact = ret / np.sqrt(traded)
    q1, q3 = np.quantile(traded, [0.25, 0.75])
    bottom, top = impact[traded <= q1], impact[traded >= q3]
    if bottom.size < 10 or top.size < 10:
        return np.nan
    mean_bottom, mean_top = bottom.mean(), top.mean()
    if mean_bottom <= 0 or mean_top <= 0:
        return np.nan
    return float(-np.log(mean_top / mean_bottom))


def _wt17_month(datasources, pool, start, end):
    raw = _wt17_read_month(datasources, pool, start, end)
    if raw.empty:
        return pd.DataFrame(columns=KEY + ["factor"])
    records = []
    for (day, instrument), frame in raw.groupby(KEY, sort=False, observed=True):
        records.append((day, instrument, _wt17_one_day(frame)))
    return pd.DataFrame(records, columns=KEY + ["factor"])


def _compute_wt17(datasources, start, end):
    pool = _pool(start, end)
    parts = [_wt17_month(datasources, pool, lo, hi) for lo, hi in _months(start, end)]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=KEY + ["factor"])


def _daily_center_rank(frame, source, target):
    numeric = pd.to_numeric(frame[source], errors="coerce")
    count = numeric.notna().groupby(frame["date"], observed=True).transform("sum")
    rank = numeric.groupby(frame["date"], observed=True).rank(method="average")
    frame[target] = (rank / count - 0.5).where(count >= 60)
    return frame


from scipy.signal import get_window
from scipy.spatial.distance import cdist


def _compute_triple(datasources, start_date, end_date):
    # BigAlpha 2026 | Traditional Quant Track | Volume Persistence, Recurrence and Spectral Rhythm Composite
    # The composite joins order-splitting persistence, price-volume recurrence and short-period volume rhythm.
    # Theory: These are complementary microstructure mechanisms rather than a fitted predictive model.
    # Construction and direction: Same-day z-scores of VOL_AC, SF14 and WATERSTONE use fixed weights 0.332293, 0.167707 and 0.500000 respectively. The weights were frozen before the 2023-2024 official validation.
    # Parameters: continuous trading 09:32-14:56; each statistic needs its stated
    # minimum count to be identifiable. These are data-quality constraints, not
    # parameters tuned on a backtest.
    # Orthogonality to volume autocorrelation: The spectral-volume component is included to diversify the persistence/recurrence block; no official-result tuning occurs inside this notebook.



    CANDIDATE = "RAW_COMBO_VOLAC_SF14_WATERSTONE"
    KEY = ["day", "instrument"]


    def _month_ranges(start_date, end_date):
        start = pd.Timestamp(start_date).normalize()
        end_exclusive = pd.Timestamp(end_date).normalize() + pd.Timedelta(days=1)
        cursor = start.replace(day=1)
        while cursor < end_exclusive:
            nxt = cursor + pd.DateOffset(months=1)
            yield max(cursor, start), min(nxt, end_exclusive)
            cursor = nxt


    def _read_month(sd, ed, source):
        members = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={"date": [sd.strftime("%Y-%m-%d"), ed.strftime("%Y-%m-%d")]},
            compression=True,
        ).df()
        if members.empty:
            return pd.DataFrame(columns=["date", "instrument", "close", "volume", "day"])
        members["day"] = pd.to_datetime(members["date"], errors="coerce")
        if getattr(members["day"].dt, "tz", None) is not None:
            members["day"] = members["day"].dt.tz_localize(None)
        members["day"] = members["day"].dt.normalize()
        members = members[["day", "instrument"]].drop_duplicates()
        instruments = members["instrument"].dropna().astype(str).unique().tolist()
        if not instruments:
            return pd.DataFrame(columns=["date", "instrument", "close", "volume", "day"])
        sql = (
            "SELECT date, instrument, close, volume "
            f"FROM {source} "
            "WHERE EXTRACT(HOUR FROM date)*100 + EXTRACT(MINUTE FROM date) >= 932 "
            "  AND EXTRACT(HOUR FROM date)*100 + EXTRACT(MINUTE FROM date) <= 1456"
        )
        df = dai.query(
            sql,
            filters={
                "date": [sd.strftime("%Y-%m-%d"), ed.strftime("%Y-%m-%d")],
                "instrument": instruments,
            },
            compression=True,
        ).df()
        if df.empty:
            return df
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        if getattr(df["date"].dt, "tz", None) is not None:
            df["date"] = df["date"].dt.tz_localize(None)
        df = df[(df["date"] >= sd) & (df["date"] < ed)].copy()
        df["day"] = df["date"].dt.normalize()
        df = df.merge(members, on=["day", "instrument"], how="inner", validate="many_to_one")
        df["close"] = pd.to_numeric(df["close"], errors="coerce")
        df["volume"] = pd.to_numeric(df["volume"], errors="coerce")
        df = df[
            np.isfinite(df["close"])
            & (df["close"] > 0)
            & np.isfinite(df["volume"])
            & (df["volume"] >= 0)
        ]
        return df.sort_values(["instrument", "day", "date"]).reset_index(drop=True)
    def _add_group_basics(df):
        df = df.copy()
        g = df.groupby(KEY, sort=False, observed=True)
        df["_pos"] = g.cumcount()
        df["_n"] = g["close"].transform("size")
        return df


    def _q90_events(df):
        q = (
            df.groupby(KEY, sort=False, observed=True)["volume"]
            .quantile(0.90)
            .rename("_q90")
            .reset_index()
        )
        events = df.merge(q, on=KEY, how="left", sort=False)
        return events[events["volume"] > events["_q90"]].copy()


    def _wt01_burstiness(df):
        events = _q90_events(_add_group_basics(df))
        events["_gap"] = events.groupby(KEY, sort=False, observed=True)["_pos"].diff()
        gap = events[np.isfinite(events["_gap"])].copy()
        stats = (
            gap.groupby(KEY, sort=False, observed=True)["_gap"]
            .agg(["size", "mean", "std"])
            .reset_index()
        )
        denom = stats["mean"] + stats["std"]
        # Existing library direction: higher event clustering was bearish.
        stats["factor"] = -(stats["std"] - stats["mean"]) / denom
        return stats[
            (stats["size"] >= 4) & np.isfinite(stats["factor"])
        ][KEY + ["factor"]]


    def _wt10_interval_entropy(df):
        events = _q90_events(_add_group_basics(df))
        events["_gap"] = events.groupby(KEY, sort=False, observed=True)["_pos"].diff()
        gap = events[np.isfinite(events["_gap"])].copy()
        if gap.empty:
            return pd.DataFrame(columns=KEY + ["factor"])
        grouped = gap.groupby(KEY, sort=False, observed=True)["_gap"]
        gap["_lo"] = grouped.transform("min")
        gap["_hi"] = grouped.transform("max")
        counts = grouped.size().rename("n").reset_index()
        flat_keys = gap[gap["_hi"] - gap["_lo"] < 1e-9][KEY].drop_duplicates()
        flat = counts.merge(flat_keys, on=KEY, how="inner")
        flat["factor"] = 0.0
        varying = gap[gap["_hi"] - gap["_lo"] >= 1e-9].copy()
        if varying.empty:
            out = flat[KEY + ["n", "factor"]]
        else:
            varying["_bin"] = np.minimum(
                (6.0 * (varying["_gap"] - varying["_lo"]) /
                 (varying["_hi"] - varying["_lo"])).astype(int),
                5,
            )
            hist = (
                varying.groupby(KEY + ["_bin"], sort=False, observed=True)
                .size()
                .rename("count")
                .reset_index()
            )
            hist["p"] = hist["count"] / hist.groupby(
                KEY, sort=False, observed=True
            )["count"].transform("sum")
            entropy = (
                (-(hist["p"] * np.log(hist["p"])))
                .groupby([hist[k] for k in KEY], sort=False)
                .sum()
                .rename("factor")
                .reset_index()
            )
            entropy["factor"] /= np.log(6.0)
            out = pd.concat(
                [flat[KEY + ["n", "factor"]], counts.merge(entropy, on=KEY, how="inner")],
                ignore_index=True,
            )
        return out[(out["n"] >= 8) & np.isfinite(out["factor"])][KEY + ["factor"]]


    def _wt19_volatility_concentration(df):
        df = _add_group_basics(df)
        df["_log_close"] = np.log(df["close"])
        df["_r2"] = df.groupby(KEY, sort=False, observed=True)["_log_close"].diff().pow(2)
        df["_block"] = np.minimum((8.0 * df["_pos"] / df["_n"]).astype(int), 7)
        rv = (
            df.groupby(KEY + ["_block"], sort=False, observed=True)["_r2"]
            .sum()
            .rename("rv")
            .reset_index()
        )
        rv["_total"] = rv.groupby(KEY, sort=False, observed=True)["rv"].transform("sum")
        rv["_share_sq"] = (rv["rv"] / rv["_total"]) ** 2
        hhi = (
            rv.groupby(KEY, sort=False, observed=True)["_share_sq"]
            .sum()
            .rename("hhi")
            .reset_index()
        )
        n = df.groupby(KEY, sort=False, observed=True)["_n"].first().rename("n").reset_index()
        out = hhi.merge(n, on=KEY, how="inner")
        # Existing library direction: concentrated event volatility was bearish.
        out["factor"] = -(out["hhi"] - 1.0 / 8.0) / (1.0 - 1.0 / 8.0)
        return out[(out["n"] >= 120) & np.isfinite(out["factor"])][KEY + ["factor"]]


    def _wt23_shrinking_new_high(df):
        df = _add_group_basics(df)
        grouped = df.groupby(KEY, sort=False, observed=True)
        running_high = grouped["close"].cummax()
        df["_renew"] = running_high > grouped["close"].transform(lambda x: x.cummax().shift(1))
        df["_volume_rank"] = grouped["volume"].rank(method="average", pct=True)
        events = df[df["_renew"]].copy()
        out = events.groupby(KEY, sort=False, observed=True).agg(
            count=("_renew", "size"),
            mean_volume_rank=("_volume_rank", "mean"),
            n=("_n", "first"),
        ).reset_index()
        out["factor"] = (1.0 - out["mean_volume_rank"]) * np.sqrt(
            out["count"] / (out["n"] - 1.0)
        )
        return out[
            (out["n"] >= 60) & (out["count"] >= 3) & np.isfinite(out["factor"])
        ][KEY + ["factor"]]


    def _volume_autocorrelation(df):
        df = _add_group_basics(df)
        df["_x"] = np.log1p(df["volume"])
        df["_y"] = df.groupby(KEY, sort=False, observed=True)["_x"].shift(-1)
        pairs = df[np.isfinite(df["_x"]) & np.isfinite(df["_y"])].copy()
        pairs["_xx"] = pairs["_x"] * pairs["_x"]
        pairs["_yy"] = pairs["_y"] * pairs["_y"]
        pairs["_xy"] = pairs["_x"] * pairs["_y"]
        stats = pairs.groupby(KEY, sort=False, observed=True).agg(
            n=("_x", "size"),
            sx=("_x", "sum"),
            sy=("_y", "sum"),
            sxx=("_xx", "sum"),
            syy=("_yy", "sum"),
            sxy=("_xy", "sum"),
        ).reset_index()
        numerator = stats["sxy"] - stats["sx"] * stats["sy"] / stats["n"]
        denominator = np.sqrt(
            (stats["sxx"] - stats["sx"] ** 2 / stats["n"])
            * (stats["syy"] - stats["sy"] ** 2 / stats["n"])
        )
        # The existing official benchmark uses the negative direction.
        stats["factor"] = -numerator / denominator
        return stats[(stats["n"] >= 60) & np.isfinite(stats["factor"])][KEY + ["factor"]]


    def _sf14_cross_coupling_frame(frame):
        close = frame["close"].to_numpy(dtype=float)
        volume = frame["volume"].to_numpy(dtype=float)
        if close.size < 80 or not np.isfinite(close).all() or close.min() <= 0 or volume.sum() <= 0:
            return np.nan
        ret = np.diff(np.log(close))
        log_volume = np.log1p(volume[1:])
        ret_std, volume_std = ret.std(), log_volume.std()
        if ret_std < 1e-12 or volume_std < 1e-9:
            return np.nan
        ret = (ret - ret.mean()) / ret_std
        log_volume = (log_volume - log_volume.mean()) / volume_std
        length = ret.size - 2
        if length < 40:
            return np.nan
        return_embedding = np.column_stack([ret[i:i + length] for i in range(3)])
        volume_embedding = np.column_stack([log_volume[i:i + length] for i in range(3)])
        recurrence = (cdist(return_embedding, volume_embedding, "euclidean") <= 0.25 * np.sqrt(3.0)).astype(np.int8)
        total = recurrence.sum()
        if total < 10:
            return np.nan
        predecessor = np.zeros_like(recurrence)
        successor = np.zeros_like(recurrence)
        predecessor[1:, 1:] = recurrence[:-1, :-1]
        successor[:-1, :-1] = recurrence[1:, 1:]
        return float(-(((recurrence > 0) & ((predecessor > 0) | (successor > 0))).sum() / total))


    def _water_stone_frame(frame):
        volume = frame["volume"].to_numpy(dtype=float)
        if volume.size < 60 or not np.isfinite(volume).all():
            return np.nan
        q75, q25, median = np.percentile(volume, 75), np.percentile(volume, 25), np.median(volume)
        volume = np.clip(volume, median - 3.0 * (q75 - q25), median + 3.0 * (q75 - q25))
        power = np.abs(np.fft.rfft((volume - volume.mean()) * get_window("hann", volume.size))) ** 2
        periods = 1.0 / (np.fft.rfftfreq(volume.size, d=1.0) + 1e-9)
        return float(power[(periods >= 2.0) & (periods <= 5.0)].sum() / (power[1:].sum() + 1e-9))


    def _raw_combo(df, weights):
        records = []
        for (day, instrument), frame in df.groupby(KEY, sort=False, observed=True):
            vol_ac = _volume_autocorrelation(frame)
            raw = {"day": day, "instrument": instrument, "vol_ac": np.nan, "sf14": np.nan, "waterstone": np.nan}
            if not vol_ac.empty:
                raw["vol_ac"] = vol_ac["factor"].iloc[0]
            raw["sf14"] = _sf14_cross_coupling_frame(frame)
            raw["waterstone"] = _water_stone_frame(frame)
            records.append(raw)
        raw = pd.DataFrame.from_records(records)
        components = list(weights)
        z = raw[["day", "instrument"]].copy()
        for name in components:
            values = raw[name]
            mean = values.groupby(raw["day"], sort=False).transform("mean")
            std = values.groupby(raw["day"], sort=False).transform(lambda x: x.std(ddof=0))
            z[name] = (values - mean) / std.replace(0.0, np.nan)
        valid = z[components].notna().all(axis=1)
        out = z.loc[valid, ["day", "instrument"]].copy()
        out["factor"] = z.loc[valid, components].mul(pd.Series(weights), axis=1).sum(axis=1)
        return out


    def _raw_combo_pair(df):
        return _raw_combo(df, {"vol_ac": 0.6645861902753413, "sf14": 0.3354138097246587})


    def _raw_combo_triple(df):
        return _raw_combo(df, {"vol_ac": 0.33229309513767065, "sf14": 0.16770690486232935, "waterstone": 0.5})


    def _compute_month(sd, ed, source):
        df = _read_month(sd, ed, source)
        if df.empty:
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        return _raw_combo_triple(df).rename(columns={"day": "date"})[
            ["date", "instrument", "factor"]
        ]


    def main(datasources, start_date, end_date):
        if not isinstance(datasources, dict) or not isinstance(datasources.get("bar1m"), str):
            raise TypeError("datasources must provide a string bar1m table name")
        source = datasources["bar1m"]
        parts = [
            _compute_month(sd, ed, source)
            for sd, ed in _month_ranges(start_date, end_date)
        ]
        out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
            columns=["date", "instrument", "factor"]
        )
        out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.normalize().astype("datetime64[ns]")
        out["factor"] = pd.to_numeric(out["factor"], errors="coerce").astype("float64")
        return (
            out.replace([np.inf, -np.inf], np.nan)
            .dropna(subset=["date", "instrument", "factor"])
            .drop_duplicates(["date", "instrument"], keep="last")
            [["date", "instrument", "factor"]]
            .sort_values(["date", "instrument"])
            .reset_index(drop=True)
        )
    return main(datasources, start_date, end_date)

def _compute_cmp01(datasources, start_date, end_date):
    # BigAlpha 2026 | Traditional Quant Track | CMP01_BA_KYLE_WATERSTONE_WT04_DECAY
    """
    因子名：CMP01_BA_KYLE_WATERSTONE_WT04_DECAY
    复合机制：BA_KYLE, WATERSTONE, WT04_DECAY
    构造：各机制按 2019--2022 官方 Barra 残差 IC 固定方向，在中证1000同日截面
    分别标准化后等权合成。2023--2024只用于门禁验证，未用于改符号或权重。
    该输出是一个复合因子，不是多个单因子的打包；仅使用 T 日官方分钟数据，
    供 T 日收盘后形成并在 T+1 使用，不读取未来收益。
    """

    import dai
    import numpy as np
    import pandas as pd

    FACTOR_ID = 'CMP01_BA_KYLE_WATERSTONE_WT04_DECAY'
    CANDIDATE = 'T_BA_KYLE__WATERSTONE__WT04_DECAY'
    SELECTED = ('BA_KYLE', 'WATERSTONE', 'WT04_DECAY')
    TRAIN_SIGNS = {'BA_KYLE': 1.0, 'WATERSTONE': 1.0, 'WT04_DECAY': 1.0}
    KEY = ["date", "instrument"]
    RAW_COLUMNS = ["open", "high", "low", "close", "volume", "amount"]


    def _ba_kyle(m):
        close = m["close"].to_numpy(dtype=float)
        volume = m["volume"].to_numpy(dtype=float)
        valid = np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume >= 0)
        close, volume = close[valid], volume[valid]
        if close.size < 20:
            return np.nan
        ret = np.abs(close[1:] / close[:-1] - 1.0)
        x = np.log1p(volume[1:])
        ok = np.isfinite(ret) & np.isfinite(x) & (volume[1:] > 0)
        ret, x = ret[ok], x[ok]
        if x.size < 10:
            return np.nan
        denom = np.sum((x - x.mean()) ** 2)
        if denom <= 1e-12:
            return np.nan
        return float(-np.sum((x - x.mean()) * (ret - ret.mean())) / denom)


    def _volume_autocorr(m):
        volume = m["volume"].to_numpy(dtype=float)
        volume = volume[np.isfinite(volume) & (volume >= 0)]
        if volume.size < 60:
            return np.nan
        logv = np.log1p(volume)
        x, y = logv[:-1], logv[1:]
        if x.std() <= 1e-12 or y.std() <= 1e-12:
            return np.nan
        return float(-np.corrcoef(x, y)[0, 1])


    def _wt01(m):
        volume = m["volume"].to_numpy(dtype=float)
        if volume.size < 60 or not np.isfinite(volume).all() or volume.sum() <= 0:
            return np.nan
        events = np.flatnonzero(volume > np.quantile(volume, 0.90))
        gaps = np.diff(events).astype(float)
        if gaps.size < 4:
            return np.nan
        mean, std = gaps.mean(), gaps.std(ddof=0)
        return float(-(std - mean) / (std + mean)) if std + mean > 1e-12 else np.nan


    def _wt10(m):
        volume = m["volume"].to_numpy(dtype=float)
        if volume.size < 60 or not np.isfinite(volume).all() or volume.sum() <= 0:
            return np.nan
        gaps = np.diff(np.flatnonzero(volume > np.quantile(volume, 0.90))).astype(float)
        if gaps.size < 8:
            return np.nan
        lo, hi = gaps.min(), gaps.max()
        if hi - lo < 1e-9:
            return 0.0
        counts, _ = np.histogram((gaps - lo) / (hi - lo), bins=6, range=(0.0, 1.0))
        p = counts[counts > 0] / counts.sum()
        return float(-(p * np.log(p)).sum() / np.log(6.0))


    def _wt19(m):
        close = m["close"].to_numpy(dtype=float)
        if close.size < 120 or not np.isfinite(close).all() or close.min() <= 0:
            return np.nan
        r2 = np.diff(np.log(close)) ** 2
        cuts = np.linspace(0, r2.size, 9).astype(int)
        rv = np.array([r2[cuts[i]:cuts[i + 1]].sum() for i in range(8)])
        total = rv.sum()
        if total <= 0:
            return np.nan
        hhi = ((rv / total) ** 2).sum()
        return float(-(hhi - 1.0 / 8.0) / (1.0 - 1.0 / 8.0))


    def _wt23(m):
        close = m["close"].to_numpy(dtype=float)
        volume = m["volume"].to_numpy(dtype=float)
        ok = np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume >= 0)
        close, volume = close[ok], volume[ok]
        if close.size < 60:
            return np.nan
        previous = np.maximum.accumulate(close)
        renew = np.r_[False, previous[1:] > previous[:-1]]
        count = int(renew.sum())
        if count < 3:
            return np.nan
        rank = pd.Series(volume).rank(method="average", pct=True).to_numpy()
        return float((1.0 - rank[renew].mean()) * np.sqrt(count / (close.size - 1.0)))


    def _sf06_volvol_rough(m: pd.DataFrame) -> float:
        c = m["close"].to_numpy(dtype=float)
        c = c[np.isfinite(c) & (c > 0)]
        if c.size < 60:
            return np.nan
        r = np.diff(np.log(c))
        r = r[np.isfinite(r)]
        full = (r.size // 6) * 6
        if full < 24:
            return np.nan
        lv = np.sqrt((r[:full].reshape(-1, 6) ** 2).sum(axis=1))
        lv = lv[np.isfinite(lv) & (lv > 0)]
        if lv.size < 4:
            return np.nan
        llv = np.log(lv)
        sd = llv.std()
        return float(np.mean(np.abs(np.diff(llv))) / sd) if sd > 1e-8 else np.nan


    def _sf09_upper_tail(m: pd.DataFrame) -> float:
        c = m["close"].to_numpy(dtype=float)
        v = m["volume"].to_numpy(dtype=float)
        ok = np.isfinite(c) & (c > 0) & np.isfinite(v) & (v >= 0)
        c, v = c[ok], v[ok]
        if c.size < 40:
            return np.nan
        r, v = np.abs(np.diff(np.log(c))), v[1:]
        n = r.size
        if n < 30:
            return np.nan
        ur = (np.argsort(np.argsort(r)) + 1.0) / (n + 1.0)
        uv = (np.argsort(np.argsort(v)) + 1.0) / (n + 1.0)
        top = ur > 0.85
        return float((uv[top] > 0.85).mean()) if top.sum() >= 4 else np.nan


    def _sf10_gumbel(m: pd.DataFrame) -> float:
        c = m["close"].to_numpy(dtype=float)
        v = m["volume"].to_numpy(dtype=float)
        if c.size < 40:
            return np.nan
        r, v = np.abs(np.diff(np.log(np.clip(c, 1e-12, None)))), v[1:]
        ok = np.isfinite(r) & np.isfinite(v) & (v > 0)
        r, v = r[ok], v[ok]
        n = r.size
        if n < 35:
            return np.nan
        ur = (np.argsort(np.argsort(r)) + 1.0) / (n + 1.0)
        uv = (np.argsort(np.argsort(v)) + 1.0) / (n + 1.0)
        values = []
        for q in (0.80, 0.85, 0.90, 0.95):
            top = ur > q
            if top.sum() < 3:
                return np.nan
            values.append((uv[top] > q).mean())
        return float(np.mean(values))


    def _sf14_cross_coupling(m: pd.DataFrame) -> float:
        from scipy.spatial.distance import cdist

        c = m["close"].to_numpy(dtype=float)
        v = m["volume"].to_numpy(dtype=float)
        if c.size < 80 or not np.isfinite(c).all() or c.min() <= 0 or v.sum() <= 0:
            return np.nan
        r, lv = np.diff(np.log(c)), np.log1p(v[1:])
        sr, sv = r.std(), lv.std()
        if sr < 1e-12 or sv < 1e-9:
            return np.nan
        r, lv = (r - r.mean()) / sr, (lv - lv.mean()) / sv
        length = r.size - 2
        if length < 40:
            return np.nan
        er = np.column_stack([r[i:i + length] for i in range(3)])
        ev = np.column_stack([lv[i:i + length] for i in range(3)])
        recurrence = (cdist(er, ev, "euclidean") <= 0.25 * np.sqrt(3.0)).astype(np.int8)
        total = recurrence.sum()
        if total < 10:
            return np.nan
        predecessor = np.zeros_like(recurrence)
        successor = np.zeros_like(recurrence)
        predecessor[1:, 1:] = recurrence[:-1, :-1]
        successor[:-1, :-1] = recurrence[1:, 1:]
        return float(-(((recurrence > 0) & ((predecessor > 0) | (successor > 0))).sum() / total))


    def _water_stone(m: pd.DataFrame) -> float:
        from scipy.signal import get_window

        v = m["volume"].to_numpy(dtype=float)
        if v.size < 60 or not np.isfinite(v).all():
            return np.nan
        q75, q25, med = np.percentile(v, 75), np.percentile(v, 25), np.median(v)
        v = np.clip(v, med - 3.0 * (q75 - q25), med + 3.0 * (q75 - q25))
        power = np.abs(np.fft.rfft((v - v.mean()) * get_window("hann", v.size))) ** 2
        periods = 1.0 / (np.fft.rfftfreq(v.size, d=1.0) + 1e-9)
        return float(power[(periods >= 2.0) & (periods <= 5.0)].sum() / (power[1:].sum() + 1e-9))


    def _wt04_decay(m: pd.DataFrame) -> float:
        c, v = m["close"].to_numpy(dtype=float), m["volume"].to_numpy(dtype=float)
        if c.size < 60 or not (np.isfinite(c).all() and np.isfinite(v).all()) or c.min() <= 0:
            return np.nan
        r, vv = np.abs(np.diff(np.log(c))), v[1:]
        events = np.flatnonzero(vv > np.quantile(vv, 0.90))
        events = events[events < r.size - 1]
        ratios = []
        for i in events:
            if r[i] >= 1e-8 and r[i + 1:min(i + 6, r.size)].size:
                ratios.append(r[i + 1:min(i + 6, r.size)].mean() / r[i])
        return float(-np.median(ratios)) if len(ratios) >= 3 else np.nan


    def _wt06_center(m: pd.DataFrame) -> float:
        v = m["volume"].to_numpy(dtype=float)
        if v.size < 60 or not np.isfinite(v).all() or v.sum() <= 0:
            return np.nan
        event = np.flatnonzero(v > np.quantile(v, 0.90))
        if event.size < 5:
            return np.nan
        return float((event * v[event]).sum() / (v[event].sum() * (v.size - 1)))


    def _wt08_fano(m: pd.DataFrame) -> float:
        v = m["volume"].to_numpy(dtype=float)
        if v.size < 60 or not np.isfinite(v).all() or v.sum() <= 0:
            return np.nan
        event = (v > np.quantile(v, 0.90)).astype(float)
        cuts = np.linspace(0, v.size, 9).astype(int)
        counts = np.array([event[cuts[i]:cuts[i + 1]].sum() for i in range(8)])
        return float(-(counts.var() / counts.mean())) if counts.mean() >= 0.5 else np.nan


    def _wt12_absorb(m: pd.DataFrame) -> float:
        h = m["high"].to_numpy(dtype=float)
        l = m["low"].to_numpy(dtype=float)
        c = m["close"].to_numpy(dtype=float)
        v = m["volume"].to_numpy(dtype=float)
        if c.size < 60 or not (np.isfinite(h).all() and np.isfinite(l).all() and np.isfinite(c).all() and np.isfinite(v).all()) or c.min() <= 0:
            return np.nan
        amp = np.where((h - l) / c >= 0, (h - l) / c, np.nan)
        event = v > np.quantile(v, 0.90)
        median = np.nanmedian(amp)
        value = np.nanmean(amp[event]) / median if event.sum() >= 5 and median >= 1e-6 else np.nan
        return float(-np.log(value)) if np.isfinite(value) and value > 0 else np.nan


    def _wt20_amount_entropy(m: pd.DataFrame) -> float:
        amount = m["amount"].to_numpy(dtype=float)
        if amount.size < 60 or not np.isfinite(amount).all() or amount.sum() <= 0:
            return np.nan
        p = amount / amount.sum()
        p = p[p > 0]
        return float(-(p * np.log(p)).sum() / np.log(p.size)) if p.size >= 30 else np.nan


    COMPONENT_FUNCTIONS = {
        'BA_KYLE': _ba_kyle,
        'VOL_AC': _volume_autocorr,
        'WT01': _wt01,
        'WT10': _wt10,
        'WT19': _wt19,
        'WT23': _wt23,
        'SF06_VOV': _sf06_volvol_rough,
        'SF09_TAIL': _sf09_upper_tail,
        'SF10_GUMBEL': _sf10_gumbel,
        'SF14_XCOUP': _sf14_cross_coupling,
        'WATERSTONE': _water_stone,
        'WT04_DECAY': _wt04_decay,
        'WT06_CENTER': _wt06_center,
        'WT08_FANO': _wt08_fano,
        'WT12_ABSORB': _wt12_absorb,
        'WT20_AMOUNT_ENT': _wt20_amount_entropy,
    }



    def _months(start_date, end_date):
        start = pd.Timestamp(start_date).normalize()
        end = pd.Timestamp(end_date).normalize() + pd.Timedelta(days=1)
        cursor = start.replace(day=1)
        while cursor < end:
            next_cursor = cursor + pd.DateOffset(months=1)
            yield max(cursor, start), min(next_cursor, end)
            cursor = next_cursor


    def _day(values):
        out = pd.to_datetime(values)
        if getattr(out.dt, "tz", None) is not None:
            out = out.dt.tz_localize(None)
        return out.dt.normalize()


    def _read_month(datasources, start, end):
        # The platform passes a datasources mapping and replaces bar1m at evaluation.
        if not isinstance(datasources, dict) or "bar1m" not in datasources:
            raise TypeError("expected datasources mapping with a 'bar1m' entry")
        source = str(datasources["bar1m"])
        members = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={"date": [start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")]},
        ).df()
        if members.empty:
            return pd.DataFrame(columns=KEY + RAW_COLUMNS)
        members["date"] = _day(members["date"])
        member_ids = members["instrument"].dropna().astype(str).unique().tolist()
        raw = dai.query(
            f"""SELECT date, instrument, open, high, low, close, volume, amount
               FROM {source}
               WHERE EXTRACT(HOUR FROM date) * 100 + EXTRACT(MINUTE FROM date) >= 932
                 AND EXTRACT(HOUR FROM date) * 100 + EXTRACT(MINUTE FROM date) <= 1456""",
            filters={
                "date": [start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")],
                "instrument": member_ids,
            },
            compression=True,
        ).df()
        if raw.empty:
            return pd.DataFrame(columns=KEY + RAW_COLUMNS)
        raw["minute"] = pd.to_datetime(raw["date"])
        raw = raw[(raw["minute"] >= start) & (raw["minute"] < end)].copy()
        raw["date"] = _day(raw["minute"])
        raw = raw.merge(members, on=KEY, how="inner")
        for column in RAW_COLUMNS:
            raw[column] = pd.to_numeric(raw[column], errors="coerce")
        return raw.sort_values(KEY + ["minute"])


    def _compute_month(datasources, start, end):
        raw = _read_month(datasources, start, end)
        records = []
        for (day, instrument), frame in raw.groupby(KEY, sort=False, observed=True):
            row = {"date": day, "instrument": instrument}
            minute = frame[RAW_COLUMNS].reset_index(drop=True)
            for name in SELECTED:
                try:
                    row[name] = TRAIN_SIGNS[name] * float(COMPONENT_FUNCTIONS[name](minute))
                except (ArithmeticError, ValueError, TypeError, IndexError):
                    row[name] = np.nan
            records.append(row)
        values = pd.DataFrame.from_records(records, columns=KEY + list(SELECTED))
        if values.empty:
            return pd.DataFrame(columns=KEY + ["factor"])
        valid = values[list(SELECTED)].notna().all(axis=1)
        values = values.loc[valid].copy()
        for name in SELECTED:
            grouped = values.groupby("date", sort=False)[name]
            mean = grouped.transform("mean")
            std = grouped.transform(lambda x: x.std(ddof=0))
            values[name] = (values[name] - mean) / std.replace(0.0, np.nan)
        values["factor"] = values[list(SELECTED)].mean(axis=1)
        return values.dropna(subset=["factor"])[KEY + ["factor"]]


    def main(datasources, start_date, end_date):
        parts = [
            _compute_month(datasources, start, end)
            for start, end in _months(start_date, end_date)
        ]
        out = (
            pd.concat(parts, ignore_index=True)
            if parts else pd.DataFrame(columns=KEY + ["factor"])
        )
        out["date"] = pd.to_datetime(out["date"]).dt.normalize().astype("datetime64[ns]")
        out["factor"] = pd.to_numeric(out["factor"], errors="coerce").astype("float64")
        return (
            out.dropna(subset=["factor"])[["date", "instrument", "factor"]]
            .drop_duplicates(KEY, keep="last")
            .sort_values(KEY)
            .reset_index(drop=True)
        )

    return main(datasources, start_date, end_date)


def _rank_z_array(values):
    numeric = pd.to_numeric(values, errors="coerce")
    rank = numeric.rank(method="average", pct=True).to_numpy(dtype=float)
    return (rank - np.nanmean(rank)) / (np.nanstd(rank, ddof=0) + 1e-12)


def _next_axis(values, previous_axes):
    residual = _rank_z_array(values)
    for axis in previous_axes:
        residual = residual - float(np.mean(residual * axis)) * axis
    return residual / (np.std(residual, ddof=0) + 1e-12)

def main(datasources, start_date, end_date):
    start = pd.Timestamp(start_date).normalize()
    end = pd.Timestamp(end_date).normalize()
    cmp01 = _compute_cmp01(datasources, start, end).rename(columns={"factor": "cmp01"})
    triple = _compute_triple(datasources, start, end).rename(columns={"factor": "triple"})
    f01 = _compute_f01(datasources, start, end).rename(columns={"factor": "f01"})
    wt17 = _compute_wt17(datasources, start, end).rename(columns={"factor": "wt17"})
    out = cmp01.merge(triple, on=KEY, how="inner", validate="one_to_one")
    out = out.merge(f01, on=KEY, how="inner", validate="one_to_one")
    out = out.merge(wt17, on=KEY, how="inner", validate="one_to_one")
    out = out.dropna(subset=["cmp01", "triple", "f01", "wt17"]).sort_values(KEY).reset_index(drop=True)
    out = _daily_center_rank(out, "f01", "f01_rank")
    out = _daily_center_rank(out, "wt17", "wt17_rank")
    out["absorption_block"] = 0.5 * (out["f01_rank"] + out["wt17_rank"])
    out["factor"] = np.nan
    for _, frame in out.groupby("date", sort=False, observed=True):
        if len(frame) < 60:
            continue
        axes = []
        axes.append(_next_axis(frame["cmp01"], axes))
        axes.append(_next_axis(frame["triple"], axes))
        axes.append(_next_axis(frame["absorption_block"], axes))
        out.loc[frame.index, "factor"] = 0.70 * axes[0] + 0.15 * axes[1] + 0.15 * axes[2]
    out["date"] = pd.to_datetime(out["date"]).dt.normalize().astype("datetime64[ns]")
    out["factor"] = pd.to_numeric(out["factor"], errors="coerce").astype("float64")
    return out.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["date", "instrument", "factor"]
    ).drop_duplicates(KEY, keep="last")[KEY + ["factor"]].sort_values(KEY).reset_index(drop=True)
